# GoogLeNet

## 0. 基本认识 GoogLeNet Overview

GoogLeNet 是 2014 年 ILSVRC 图像分类竞赛中的代表性卷积神经网络，对应论文是 *Going Deeper with Convolutions*，通常也叫 Inception v1。

这份笔记现在按两个层次理解：

- 原版模型：面向 ImageNet，输入一般是 $3 \times 224 \times 224$，输出 1000 类。
- 本地源码版：`GoogLeNet-1` 用猫狗图片做二分类，输入是 RGB 三通道，最后 `Linear(1024, 2)` 输出猫和狗两类。

核心思路可以概括为：

- 用 Inception 模块并联多个卷积和池化分支，同时提取不同尺度的图像特征。
- 用 $1 \times 1$ 卷积控制通道数，减少后续 $3 \times 3$、$5 \times 5$ 卷积的计算量。
- 用全局平均池化把 $1024 \times 7 \times 7$ 特征压成 $1024$ 维，再接分类层。
- 本地源码没有实现辅助分类器、Dropout 和 BatchNorm，结构比论文/官方实现更简化。

简单理解：这份代码保留了 GoogLeNet 的 Inception 主干，把最后分类头改成猫狗二分类。

---

## 1. Inception 模块 Inception Module

### 1.1 源码中的模块参数

`model.py` 中定义了：

```python
class Inception(nn.Module):
    def __init__(self, in_channels, c1, c2, c3, c4):
        ...
```

这里的参数含义是：

| 参数 | 对应分支 | 含义 |
| --- | --- | --- |
| `in_channels` | 所有分支 | 输入特征图通道数 |
| `c1` | 分支 1 | $1 \times 1$ 卷积输出通道数 |
| `c2=(a,b)` | 分支 2 | 先 $1 \times 1$ 降到 `a` 通道，再 $3 \times 3$ 输出 `b` 通道 |
| `c3=(a,b)` | 分支 3 | 先 $1 \times 1$ 降到 `a` 通道，再 $5 \times 5$ 输出 `b` 通道 |
| `c4` | 分支 4 | 最大池化后接 $1 \times 1$ 卷积输出通道数 |

四个分支最后按通道维度拼接：

```python
return torch.cat((p1, p2, p3, p4), dim=1)
```

PyTorch 图像张量常用格式是 `(B, C, H, W)`，所以 `dim=1` 表示按通道拼接。

### 1.2 四条路线

源码中的 Inception 可以写成：

```text
路线1: 1x1 Conv -> ReLU
路线2: 1x1 Conv -> ReLU -> 3x3 Conv, padding=1 -> ReLU
路线3: 1x1 Conv -> ReLU -> 5x5 Conv, padding=2 -> ReLU
路线4: 3x3 MaxPool, stride=1, padding=1 -> 1x1 Conv -> ReLU
输出: cat([p1, p2, p3, p4], dim=1)
```

`padding=1` 的 $3 \times 3$ 卷积、`padding=2` 的 $5 \times 5$ 卷积，以及 `stride=1, padding=1` 的池化分支，都会保持空间尺寸不变。这样四个分支才能正常拼接。

### 1.3 输出通道数计算

Inception 模块的输出通道数为：

$$
C_{out} = c1 + c2[1] + c3[1] + c4
$$

以源码中的第一个 Inception 为例：

```python
Inception(192, 64, (96, 128), (16, 32), 32)
```

输出通道数为：

$$
64 + 128 + 32 + 32 = 256
$$

所以输入如果是 $192 \times 28 \times 28$，输出就是 $256 \times 28 \times 28$。

---

## 2. 网络结构 Network Architecture

### 2.1 源码主线

`GoogLeNet` 在 `model.py` 中被拆成 `b1` 到 `b5` 五个顺序模块，前向传播很直接：

```python
x = self.b1(x)
x = self.b2(x)
x = self.b3(x)
x = self.b4(x)
x = self.b5(x)
```

本地源码版输入是 RGB 图片，形状为 `(B, 3, 224, 224)`；输出是 `(B, 2)`，对应猫和狗两类。

### 2.2 b1 到 b5 结构

| 模块 | 主要结构 | 输出尺寸 |
| --- | --- | --- |
| `b1` | `Conv2d(3,64,7,stride=2,padding=3)` + ReLU + `MaxPool2d(3,stride=2,padding=1)` | $64 \times 56 \times 56$ |
| `b2` | `Conv2d(64,64,1)` + ReLU + `Conv2d(64,192,3,padding=1)` + ReLU + MaxPool | $192 \times 28 \times 28$ |
| `b3` | Inception 3a + Inception 3b + MaxPool | $480 \times 14 \times 14$ |
| `b4` | Inception 4a 到 4e + MaxPool | $832 \times 7 \times 7$ |
| `b5` | Inception 5a + Inception 5b + AdaptiveAvgPool + Flatten + Linear | $2$ |

整体尺寸变化可以记成：

```text
3 x 224 x 224
-> 64 x 56 x 56
-> 192 x 28 x 28
-> 480 x 14 x 14
-> 832 x 7 x 7
-> 1024 x 1 x 1
-> 2
```

### 2.3 Inception 配置表

源码中的 Inception 配置如下：

| 位置 | 写法 | 输出通道数 |
| --- | --- | --- |
| 3a | `Inception(192, 64, (96,128), (16,32), 32)` | 256 |
| 3b | `Inception(256, 128, (128,192), (32,96), 64)` | 480 |
| 4a | `Inception(480, 192, (96,208), (16,48), 64)` | 512 |
| 4b | `Inception(512, 160, (112,224), (24,64), 64)` | 512 |
| 4c | `Inception(512, 128, (128,256), (24,64), 64)` | 512 |
| 4d | `Inception(512, 112, (128,288), (32,64), 64)` | 528 |
| 4e | `Inception(528, 256, (160,320), (32,128), 128)` | 832 |
| 5a | `Inception(832, 256, (160,320), (32,128), 128)` | 832 |
| 5b | `Inception(832, 384, (192,384), (48,128), 128)` | 1024 |

这和原版 GoogLeNet 的主干通道变化基本一致，但最后分类层从 1000 类改成了 2 类。

---

## 3. 尺寸和参数计算 Parameter Calculation

### 3.1 卷积和池化输出尺寸

卷积或池化后的空间尺寸按下面公式计算：

$$
O = \left\lfloor \frac{I + 2P - K}{S} \right\rfloor + 1
$$

其中：

- $I$：输入尺寸。
- $P$：padding 大小。
- $K$：卷积核或池化核大小。
- $S$：stride 步幅。
- $O$：输出尺寸。

### 3.2 b1 的尺寸计算

第一层卷积输入 $224 \times 224$，卷积核 $7 \times 7$，stride=2，padding=3：

$$
O = \left\lfloor \frac{224 + 2 \times 3 - 7}{2} \right\rfloor + 1 = 112
$$

所以卷积后空间尺寸为 $112 \times 112$，通道数变为 64。

接着最大池化使用 $3 \times 3$，stride=2，padding=1：

$$
O = \left\lfloor \frac{112 + 2 \times 1 - 3}{2} \right\rfloor + 1 = 56
$$

所以 `b1` 输出为 $64 \times 56 \times 56$。这里能得到 56，是因为源码的 MaxPool 明确写了 `padding=1`。

### 3.3 主干空间尺寸变化

后续每次 MaxPool 都使用 `kernel_size=3, stride=2, padding=1`，因此空间尺寸大致按下面变化：

```text
224 -> 112 -> 56 -> 28 -> 14 -> 7 -> 1
```

其中最后的 $7 \rightarrow 1$ 来自：

```python
nn.AdaptiveAvgPool2d((1, 1))
```

AdaptiveAvgPool 不要求手写池化核大小，它会自动把每个通道压缩成 $1 \times 1$。

### 3.4 Inception 通道计算

以 `Inception(528, 256, (160, 320), (32, 128), 128)` 为例：

$$
C_{out} = 256 + 320 + 128 + 128 = 832
$$

所以它会把输入的 $528$ 通道变成 $832$ 通道，但空间尺寸保持不变。

### 3.5 最后分类层

`b5` 末尾是：

```python
nn.AdaptiveAvgPool2d((1, 1))
nn.Flatten()
nn.Linear(1024, 2)
```

尺寸变化为：

```text
1024 x 7 x 7 -> 1024 x 1 x 1 -> 1024 -> 2
```

这里的 `2` 对应猫狗二分类。训练时用 `CrossEntropyLoss`，模型最后直接输出 logits，不需要手动加 Softmax。

---

## 4. 关键思想 Key Ideas

### 4.1 多尺度特征提取

Inception 模块同时使用 $1 \times 1$、$3 \times 3$、$5 \times 5$ 和池化分支，相当于让网络在同一层里观察不同感受野。

直观理解：

- $1 \times 1$ 分支关注通道组合。
- $3 \times 3$ 分支关注较小邻域的局部纹理。
- $5 \times 5$ 分支关注更大范围的上下文。
- 池化分支保留局部区域中最明显的响应。

最后把这些特征拼接起来，让后续层自己选择有用信息。

### 4.2 用 $1 \times 1$ 卷积做瓶颈层

源码中所有 $3 \times 3$ 和 $5 \times 5$ 分支前面都有 $1 \times 1$ 卷积。它不是为了改变宽高，而是为了先压缩通道数。

例如第一个 Inception 的 $5 \times 5$ 分支是：

```text
192 -> 16 -> 32
```

如果直接在 192 通道上做 $5 \times 5$ 卷积，计算量会很大；先降到 16 通道后再做 $5 \times 5$，参数量会小很多。

### 4.3 本地源码和原版的差别

这份源码保留了 Inception 主干，但做了简化：

| 项目 | 原版 GoogLeNet | 本地源码版 |
| --- | --- | --- |
| 任务 | ImageNet 1000 类 | 猫狗 2 类 |
| 输入 | RGB 图像 | RGB 图像，resize 到 $224 \times 224$ |
| 辅助分类器 | 有 | 没有 |
| Dropout | 有 | 没有 |
| BatchNorm | 原版没有，现代实现常加 | 没有 |
| 分类头 | Global Average Pooling + 1000 类 | AdaptiveAvgPool + `Linear(1024,2)` |

因此看这份代码时，不要把辅助分类器 loss 或 Dropout 写进训练流程。

### 4.4 参数初始化

源码对卷积层使用 Kaiming 初始化：

```python
nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity='relu')
```

这和 ReLU 激活比较匹配。全连接层理论上也想初始化，但当前代码的 `Linear` 初始化分支缩进有问题，实际不会执行，注意点里单独说明。

---

## 5. 源码实战版 Source Implementation

### 5.1 数据划分

`data_partitioning.py` 从 `data_cat_dog` 读取类别文件夹，并复制到：

```text
data/train/cat
data/train/dog
data/test/cat
data/test/dog
```

脚本里 `split_rate = 0.1`，意思是先把原始数据按 9:1 分成训练目录和测试目录。当前目录统计结果是：

| 目录 | 数量 |
| --- | ---: |
| `data/train/cat` | 4491 |
| `data/train/dog` | 4500 |
| `data/test/cat` | 499 |
| `data/test/dog` | 499 |

`model_train.py` 又会对 `data/train` 做一次 `random_split`，按 8:2 拆成训练集和验证集。所以整体上大致是：原始数据的 72% 用于训练，18% 用于验证，10% 用于测试。

### 5.2 数据预处理

训练和测试都使用 `ImageFolder` 读取图片，并使用同一套 transform：

```python
normalize = transforms.Normalize([0.162, 0.151, 0.138], [0.058, 0.052, 0.048])
transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])
```

这说明模型输入是 `(B, 3, 224, 224)`。`ImageFolder` 会根据子文件夹名生成标签，当前文件夹是 `cat` 和 `dog`，测试代码中对应：

```python
classes = ['猫', '狗']
```

### 5.3 训练流程

`model_train.py` 的训练流程是：

1. 构建 `GoogLeNet(Inception)`。
2. 读取 `data/train`，并按 8:2 划分训练集和验证集。
3. 使用 `DataLoader(batch_size=32, shuffle=True)`。
4. 自动选择 `cuda` 或 `cpu`。
5. 使用 Adam 优化器，学习率为 0.001。
6. 使用 `nn.CrossEntropyLoss()`。
7. 每个 epoch 先训练，再验证，统计 loss 和 accuracy。
8. 如果验证集准确率更高，就保存当前权重到 `best_model_wts`。
9. 训练结束后保存最佳权重，并画 loss / accuracy 曲线。

主函数中训练轮数是：

```python
train_model_process(GoogLeNet, train_data, val_data, num_epochs=50)
```

### 5.4 测试和单张图片预测

`model_test.py` 做了两类测试：

1. 读取 `data/test`，计算整体测试准确率。
2. 遍历测试集，打印每张图的预测值和真实值。
3. 额外读取 `1.jfif`，做单张图片预测。

测试时使用：

```python
with torch.no_grad():
    model.eval()
    output = model(image)
    pre_lab = torch.argmax(output, dim=1)
```

这是正确的测试写法：关闭梯度计算，使用评估模式，然后取 logits 中最大值对应的类别。

### 5.5 均值和方差脚本

`mean_std.py` 用来遍历 `data_cat_dog`，统计图片像素的均值和方差。训练和测试中写死的归一化参数来自这类统计结果。

需要注意：`transforms.Normalize(mean, std)` 第二个参数应该是标准差 `std`，不是方差 `variance`。如果重新计算数据集统计量，要确认自己传入的是标准差。

---

## 6. 注意点 Common Pitfalls

### 6.1 `summary` 输入通道写错

`model.py` 最后写的是：

```python
print(summary(model, (1, 224, 224)))
```

但模型第一层是：

```python
nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2, padding=3)
```

所以这份猫狗源码应该用 RGB 三通道输入，summary 更合理的写法是：

```python
summary(model, (3, 224, 224))
```

否则会出现输入通道数和第一层卷积不匹配的问题。

### 6.2 保存路径和加载路径不一致

训练脚本保存的是绝对路径：

```python
torch.save(best_model_wts, "C:/Users/86159/Desktop/GoogLeNet-1/best_model.pth")
```

测试脚本加载的是当前目录相对路径：

```python
model.load_state_dict(torch.load('best_model.pth'))
```

如果重新训练后想直接测试，最好把保存和加载路径统一，否则测试脚本可能加载的不是刚训练出来的模型。

### 6.3 验证阶段建议加 `torch.no_grad()`

训练脚本中验证阶段调用了 `model.eval()`，但没有包在 `with torch.no_grad():` 中。结果一般不受影响，但会多保留计算图，浪费显存和内存。

更稳的写法是：

```python
model.eval()
with torch.no_grad():
    output = model(b_x)
```

### 6.4 全连接层初始化分支缩进有问题

源码中 `elif isinstance(m, nn.Linear)` 缩进在 `if isinstance(m, nn.Conv2d)` 内部，导致 Linear 层初始化分支实际不会执行。

如果要初始化全连接层，应让它和卷积层判断对齐：

```python
if isinstance(m, nn.Conv2d):
    ...
elif isinstance(m, nn.Linear):
    ...
```

当前代码卷积层的 Kaiming 初始化可以执行，但全连接层不会走到手写的 normal 初始化。

### 6.5 均值和标准差统计要小心

`mean_std.py` 中 `total_pixels += normalized_image_array.size` 会把通道数也算进去。对于 RGB 图片，如果要分别统计每个通道的均值，通常应该累计 `H * W`，而不是 `H * W * C`。

另外脚本打印的是 `Variance`，但 `Normalize` 需要的是标准差。实际复用统计结果时，要确认是否需要开方：

$$
std = \sqrt{variance}
$$

### 6.6 类别顺序要和文件夹名一致

`ImageFolder` 会按文件夹名排序生成类别编号。当前目录是 `cat`、`dog`，通常对应：

```text
cat -> 0
dog -> 1
```

测试代码写 `classes = ['猫', '狗']` 正好匹配这个顺序。如果以后文件夹名或类别变了，这个列表也要同步改。

### 6.7 本地运行环境需要依赖

源码需要 `torch`、`torchvision`、`torchsummary`、`pandas`、`matplotlib`、`PIL` 等依赖。运行前要先确认当前 Python 环境已经安装这些包。

---

## 7. 总结 Summary

- 这份 `GoogLeNet-1` 源码实现的是猫狗二分类版 GoogLeNet，输入是 $3 \times 224 \times 224$，输出是 2 类。
- `Inception` 模块有四条分支：$1 \times 1$、$1 \times 1 \rightarrow 3 \times 3$、$1 \times 1 \rightarrow 5 \times 5$、MaxPool $\rightarrow 1 \times 1$。
- 每个 Inception 的输出通道数等于四个分支输出通道数之和，空间尺寸通过 padding 保持不变。
- 主干结构是 `b1 -> b2 -> b3 -> b4 -> b5`，尺寸大致从 $224$ 降到 $56, 28, 14, 7$，最后自适应平均池化到 $1 \times 1$。
- 数据流程是先用 `data_partitioning.py` 把 `data_cat_dog` 分成 `data/train` 和 `data/test`，再在训练脚本里把 `data/train` 按 8:2 划分训练集和验证集。
- 训练使用 Adam、学习率 0.001、交叉熵损失，保存验证集准确率最高的权重。
- 这份源码没有辅助分类器、Dropout 和 BatchNorm；复习时重点看 Inception 分支、通道拼接、AdaptiveAvgPool 和二分类训练流程。
- 代码里需要特别注意 summary 输入通道、模型保存/加载路径、验证阶段 `no_grad`、Linear 初始化缩进、均值标准差统计这几个问题。

***********